In [74]:
import pandas as pd

path = "../data/"

races = pd.read_csv(path + "races.csv", na_values=["\\N"])
results = pd.read_csv(path + "results.csv", na_values=["\\N"])
circuits = pd.read_csv(path + "circuits.csv", na_values=["\\N"])
quali = pd.read_csv(path + "qualifying.csv", na_values=["\\N"])
status = pd.read_csv(path + "status.csv", na_values=["\\N"])
drivers = pd.read_csv(path + "drivers.csv", na_values=["\\N"])

# Filter races for 2014-2024 seasons (post introduction of mandatory Hybrid engine)
hybridEra = races[(races["year"] >= 2014) & (races["year"] <= 2024)]
# Filter results based on races present in filtered hybridEra races table
hybridResults = results[results["raceId"].isin(hybridEra["raceId"])]
#hybridResults
hybridQuali = quali[quali["raceId"].isin(hybridEra["raceId"])]

In [2]:
hybridResults = hybridResults.merge(status, on="statusId", how="left")
#print(statusCheck["status"].value_counts())
hybridResults[["raceId", "status"]]

,raceId,status
0,900,Finished
1,900,Finished
2,900,Finished
3,900,Finished
4,900,Finished
...,...,...
4621,1144,+1 Lap
4622,1144,Engine
4623,1144,Collision damage
4624,1144,Engine


In [10]:
hybridMerged = hybridResults.merge(hybridEra[["raceId", "year", "circuitId", "name"]], on="raceId", how="left", validate="many_to_one")
hybridMerged.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,status,year,circuitId,name
0,22130,900,3,131,6.0,3,1.0,1,1,25.0,...,5578710.0,19.0,1.0,1:32.478,206.436,1,Finished,2014,1,Australian Grand Prix
1,22131,900,825,1,20.0,4,2.0,2,2,18.0,...,5605487.0,49.0,6.0,1:33.066,205.131,1,Finished,2014,1,Australian Grand Prix
2,22132,900,18,1,22.0,10,3.0,3,3,15.0,...,5608737.0,39.0,5.0,1:32.917,205.460,1,Finished,2014,1,Australian Grand Prix
3,22133,900,4,6,14.0,5,4.0,4,4,12.0,...,5613994.0,57.0,7.0,1:33.186,204.867,1,Finished,2014,1,Australian Grand Prix
4,22134,900,822,3,77.0,15,5.0,5,5,10.0,...,5626349.0,56.0,3.0,1:32.616,206.128,1,Finished,2014,1,Australian Grand Prix


In [ ]:
print("missing years: ", hybridMerged["year"].isna().sum(),
      "\nmissing circuits: ", hybridMerged["circuitId"].isna().sum(),
      "\nmin year: ", hybridMerged["year"].min(),
      "\nmax year: ", hybridMerged["year"].max())

missing years:  0 
missing circuits:  0 
min year:  2014 
max year:  2024


In [17]:
qualiPos = hybridQuali[["raceId", "driverId", "position"]]
qualiPos = qualiPos.rename(columns={"position": "qualiPosition"})

In [18]:
f1Data = hybridMerged.merge(qualiPos, on=["raceId", "driverId"], how="left", validate="one_to_one")

In [25]:
print(hybridMerged.shape, f1Data.shape,
      "\nmissing: ", f1Data["qualiPosition"].isna().sum())


(4626, 22) (4626, 23) 
missing:  16


In [29]:
missing = f1Data[f1Data["qualiPosition"].isna() == True]
missing[["raceId", "year", "name", "driverId", "grid", "position", "positionText"]]

,raceId,year,name,driverId,grid,position,positionText
390,918,2014,Abu Dhabi Grand Prix,817,20,4.0,4
394,918,2014,Abu Dhabi Grand Prix,20,19,8.0,8
444,927,2015,Malaysian Grand Prix,829,0,NaN,W
484,929,2015,Bahrain Grand Prix,18,20,NaN,W
543,932,2015,Canadian Grand Prix,18,20,NaN,R
699,941,2015,Russian Grand Prix,832,20,NaN,R
932,954,2016,Canadian Grand Prix,825,22,16.0,16
1003,957,2016,British Grand Prix,828,22,NaN,R
1055,960,2016,Belgian Grand Prix,4,22,7.0,7
1088,961,2016,Italian Grand Prix,839,22,18.0,18


In [58]:
compare = f1Data[(f1Data["qualiPosition"].notna()) & (f1Data["grid"] > 0)]
print(len(compare),
      "\nmatching: ", (((compare["qualiPosition"] == compare["grid"]).sum())/compare.shape[0])*100, 
      "\ndifferent: ", (((compare["qualiPosition"] != compare["grid"]).sum())/compare.shape[0])*100)


4533 
matching:  69.62276637988087 
different:  30.377233620119128


In [57]:
print(len(f1Data))
print(len(compare))

4626
4533


In [45]:
print(f1Data["qualiPosition"].isna().sum(),
      "\n", (f1Data["grid"] == 0).sum(),
      "\n", (f1Data["qualiPosition"].notna() & f1Data["grid"] > 0).sum())

16 
 79 
 2296


In [49]:
print(f1Data[["grid", "qualiPosition"]].dtypes)
print(f1Data["grid"].describe())
print(f1Data["grid"].isna().sum())


grid               int64
qualiPosition    float64
dtype: object
count    4626.000000
mean       10.349978
std         5.929102
min         0.000000
25%         5.000000
50%        10.000000
75%        15.000000
max        22.000000
Name: grid, dtype: float64
0


In [55]:
print("valid: ", f1Data["qualiPosition"].notna().sum())
print("grid>0: ", (f1Data["grid"] > 0).sum())
print("both: ", (f1Data["qualiPosition"].notna() & (f1Data["grid"] > 0)).sum())

valid:  4610
grid>0:  4547
both:  4533


In [60]:
analysisData = f1Data[f1Data["qualiPosition"].notna() & f1Data["position"].notna()]
print("f1Data: ", len(f1Data), 
      "\nanalysisData: ", len(analysisData),
      "\nmissing: ", len(f1Data)-len(analysisData))

f1Data:  4626 
analysisData:  3895 
missing:  731


In [61]:
print(analysisData[["qualiPosition", "position", "grid"]].dtypes,
      "\n", analysisData[["qualiPosition", "position", "grid"]].isna().sum())

qualiPosition    float64
position         float64
grid               int64
dtype: object 
 qualiPosition    0
position         0
grid             0
dtype: int64


In [71]:
analysisData["positionDiff"] = (analysisData["qualiPosition"] - analysisData["position"])
analysisData[["raceId", "number", "qualiPosition", "position", "positionDiff"]].head(10)

,raceId,number,qualiPosition,position,positionDiff
0,900,6.0,3.0,1.0,2.0
1,900,20.0,4.0,2.0,2.0
2,900,22.0,11.0,3.0,8.0
3,900,14.0,5.0,4.0,1.0
4,900,77.0,10.0,5.0,5.0
5,900,27.0,7.0,6.0,1.0
6,900,7.0,12.0,7.0,5.0
7,900,25.0,6.0,8.0,-2.0
8,900,26.0,8.0,9.0,-1.0
9,900,11.0,16.0,10.0,6.0


In [75]:
dupes = analysisData.duplicated(subset=["raceId", "driverId"])
dupes.sum()

np.int64(0)